# All In Currently Active

In [ ]:
%load https://raw.githubusercontent.com/razorrrzz/Pre-structure/main/ProgressBar.py

In [4]:
# ADD PROGRESS BAR (universal)
import time
import threading
import uuid
from datetime import datetime
from IPython.display import display, HTML, clear_output
from IPython.core.interactiveshell import InteractiveShell

# Get the shell instance
shell = InteractiveShell.instance()

# ════════════════════════════════════════════════════════════
# 1. CLEANUP (Prevent duplicate timers if you run this twice)
# ════════════════════════════════════════════════════════════
for event_type in ['pre_run_cell', 'post_run_cell']:
    if event_type in shell.events.callbacks:
        current_callbacks = shell.events.callbacks[event_type][:]
        for callback in current_callbacks:
            if 'timer' in callback.__name__ or 'progress' in callback.__name__:
                shell.events.unregister(event_type, callback)

# ════════════════════════════════════════════════════════════
# 2. GLOBAL STATE
# ════════════════════════════════════════════════════════════
_timer_running = False
_start_timestamp = 0
_display_id = None
_thread = None

# CSS for the animated bar (Indeterminate/Loading style)
# CHANGED: Replaced specific gray color with 'inherit' + opacity 
# to support both Dark and Light themes automatically.
_PROGRESS_CSS = """
<style>
@keyframes gradient-animation {
    0% { background-position: 0% 50%; }
    50% { background-position: 100% 50%; }
    100% { background-position: 0% 50%; }
}
.live-timer-bar {
    height: 4px;
    width: 100%;
    background: linear-gradient(270deg, #4caf50, #8bc34a, #cddc39);
    background-size: 200% 200%;
    animation: gradient-animation 2s ease infinite;
    border-radius: 2px;
    margin-bottom: 5px;
}
.live-timer-text {
    font-family: monospace;
    font-size: 12px;
    color: inherit;     /* Adapts to theme (Black or White) */
    opacity: 0.8;       /* Makes it slightly dimmed like original #666 */
}
</style>
"""

# ════════════════════════════════════════════════════════════
# 3. BACKGROUND THREAD FUNCTION
# ════════════════════════════════════════════════════════════
def update_progress_bar(display_id, start_time):
    """Updates the display handle every 0.1s while code runs."""
    while _timer_running:
        elapsed = time.time() - start_time
        
        # Create HTML content: CSS Bar + Text Timer
        html_content = f"""
        {_PROGRESS_CSS}
        <div class="live-timer-text">
            <div class="live-timer-bar"></div>
            Running... ⏱️ {elapsed:.1f}s
        </div>
        """
        
        # Update the specific output area
        try:
            display(HTML(html_content), display_id=display_id, update=True)
        except:
            break
            
        time.sleep(0.1)

# ════════════════════════════════════════════════════════════
# 4. HOOKS
# ════════════════════════════════════════════════════════════

def start_progress_hook(*args):
    global _timer_running, _start_timestamp, _display_id, _thread
    
    _timer_running = True
    _start_timestamp = time.time()
    
    # Generate a unique ID for this cell's output
    _display_id = str(uuid.uuid4())
    
    # Create the initial display object
    display(HTML(f"{_PROGRESS_CSS}<div class='live-timer-text'>Starting...</div>"), display_id=_display_id)
    
    # Start the background thread
    _thread = threading.Thread(target=update_progress_bar, args=(_display_id, _start_timestamp))
    _thread.daemon = True # Ensure thread dies if notebook crashes
    _thread.start()

def stop_progress_hook(*args):
    global _timer_running
    
    # Stop the thread loop
    _timer_running = False
    
    # Calculate final stats
    end_time = time.time()
    elapsed = end_time - _start_timestamp
    now_str = datetime.now().strftime("%I:%M:%S")
    
    # Wait briefly for thread to finish (optional)
    if _thread:
        _thread.join(timeout=0.2)
    
    # CHANGED: 
    # 1. color: inherit (Adapts to dark/light text)
    # 2. border-top: rgba(127,127,127,0.3) (Adapts border to look good on both backgrounds)
    final_html = f"""
    <div style="font-family: monospace; font-size: 12px; color: inherit; border-top: 1px solid rgba(127, 127, 127, 0.3); margin-top: 5px; padding-top: 2px;">
        ✅ {now_str}  (⏱️ {elapsed:.2f}s)
    </div>
    """
    
    # Update the display one last time
    try:
        display(HTML(final_html), display_id=_display_id, update=True)
    except:
        pass

# ════════════════════════════════════════════════════════════
# 5. REGISTER
# ════════════════════════════════════════════════════════════
shell.events.register('pre_run_cell', start_progress_hook)
shell.events.register('post_run_cell', stop_progress_hook)

print("✅ Live Progress Bar & Timer Enabled!")

✅ Live Progress Bar & Timer Enabled!


In [ ]:
https://github.com/razorrrzz/Pre-structurerogressBar.py

In [2]:
# CELL 1: IMPORTS & VISUAL HELPERS
# ═══════════════════════════════════════════════════════════════════════════════

import pandas as pd
import requests
import concurrent.futures
import time
from datetime import datetime, timezone, timedelta
from IPython.display import display, HTML
from jinja2 import Template
import inspect

def get_funding_color_dynamic(cell_data, row_float_values, dark_mode=False):
    if cell_data is None:
        text_color = '#666666' if dark_mode else '#999999'
        return 'transparent', text_color, '-'

    val, interval = cell_data
    try:
        if not row_float_values:
            text_color = '#888888'
            return 'transparent', text_color, f'{val:.4f}%'

        max_abs = max(abs(v) for v in row_float_values)
        if max_abs == 0: max_abs = 0.0001
        intensity = min(abs(val) / max_abs, 1.0)
        intensity = 0.3 + (intensity * 0.7)

        if val >= 0: # GREEN (POSITIVE RATES)
            if dark_mode:
                r, g, b = int(136 - intensity * 136), int(204 + intensity * 51), int(136 - intensity * 71)
            else:
                r, g, b = int(102 - intensity * 102), int(170 - intensity * 70), int(102 - intensity * 102)
        else: # RED (NEGATIVE RATES)
            if dark_mode:
                r, g, b = int(204 + intensity * 51), int(136 - intensity * 68), int(136 - intensity * 68)
            else:
                r, g, b = int(170 - intensity * 31), int(102 - intensity * 102), int(102 - intensity * 102)
        
        text_color = f'rgb({r},{g},{b})'
        
        # ADD THIS SECTION FOR COLORED INTERVALS
        # Match timeline colors
        interval_color = get_interval_color(interval, dark_mode)

        formatted_str = f'{val:.4f}%<sup style="font-size:9px; margin-left:2px; color:{interval_color}; font-weight:600;">{interval}</sup>'
        
        if interval:
            if dark_mode:
                # Dark mode colors (matching timeline)
                if '1h' in interval.lower() or '1H' in interval:
                    interval_color = '#4dabf7'  # Blue
                elif '4h' in interval.lower() or '4H' in interval:
                    interval_color = '#51cf66'  # Green
                elif '8h' in interval.lower() or '8H' in interval:
                    interval_color = '#ffd43b'  # Yellow
            else:
                # Light mode colors (matching timeline)
                if '1h' in interval.lower() or '1H' in interval:
                    interval_color = '#1976d2'  # Blue
                elif '4h' in interval.lower() or '4H' in interval:
                    interval_color = '#2e7d32'  # Green
                elif '8h' in interval.lower() or '8H' in interval:
                    interval_color = '#f57c00'  # Orange
        
        # Format with colored interval
        formatted_str = f'{val:.4f}%<sup style="font-size:9px; margin-left:2px; color:{interval_color}; font-weight:600;">{interval}</sup>'
        
        return 'transparent', text_color, formatted_str
    except Exception:
        return 'transparent', '#666666', 'Err'

print("✅ Imports loaded.")

✅ Imports loaded.


In [3]:
# CELL 2: THE ENGINE (AUTO-DETECTION ENABLED)
# ═══════════════════════════════════════════════════════════════════════════════

def get_binance_intervals_map():
    """Fetch funding intervals from Binance fundingInfo endpoint"""
    intervals = {}
    try:
        r = requests.get("https://fapi.binance.com/fapi/v1/fundingInfo", timeout=10)
        data = r.json()
        for item in data:
            symbol = item['symbol']
            hours = item.get('fundingIntervalHours', 8)
            intervals[symbol] = f"{hours}h"
    except Exception as e:
        print(f"⚠️ Binance intervals error: {e}")
    return intervals


def fetch_raw_binance():
    """
    Uses lastFundingRate from API - this is the accurate settled rate.
    Manual calculation is inaccurate because Binance uses TWAP over 8 hours.
    """
    rates = {}
    try:
        intervals_map = get_binance_intervals_map()
        r = requests.get("https://fapi.binance.com/fapi/v1/premiumIndex", timeout=10)
        
        for item in r.json():
            symbol = item['symbol']
            if symbol.endswith('USDT') and 'USDC' not in symbol:
                coin = symbol.replace('USDT', '')
                
                # ✅ Use API-provided rate directly (accurate!)
                rate = float(item['lastFundingRate']) * 100
                
                interval = intervals_map.get(symbol, '8h')
                rates[coin] = (rate, interval)
                
    except Exception as e:
        print(f"⚠️ Binance error: {e}")
    return rates

##
def get_bybit_intervals_map():
    """Fetch funding intervals from Bybit instruments endpoint"""
    intervals = {}
    try:
        r = requests.get("https://api.bybit.com/v5/market/instruments-info?category=linear", timeout=10)
        data = r.json()
        if data.get('retCode') == 0:
            for item in data['result']['list']:
                funding_interval = item.get('fundingInterval', 480)
                
                if isinstance(funding_interval, int):
                    h = funding_interval // 60
                elif isinstance(funding_interval, str):
                    h = int(funding_interval.split(':')[0])
                else:
                    h = 8
                    
                intervals[item['symbol']] = f"{h}h"
    except Exception as e:
        print(f"⚠️ Bybit intervals error: {e}")
    return intervals


def fetch_raw_bybit():
    """
    Bybit fundingRate is the predicted rate for next settlement.
    """
    rates = {}
    try:
        intervals_map = get_bybit_intervals_map()
        r = requests.get("https://api.bybit.com/v5/market/tickers?category=linear", timeout=10)
        data = r.json()
        if data.get('retCode') == 0:
            for item in data['result']['list']:
                if item['symbol'].endswith('USDT'):
                    coin = item['symbol'].replace('USDT', '')
                    # ✅ API-provided predicted rate
                    rate = float(item['fundingRate']) * 100
                    interval = intervals_map.get(item['symbol'], '8h')
                    rates[coin] = (rate, interval)
    except Exception as e:
        print(f"⚠️ Bybit error: {e}")
    return rates

##
def get_kucoin_intervals_map():
    """Fetch funding intervals from KuCoin contracts endpoint"""
    intervals = {}
    try:
        r = requests.get("https://api-futures.kucoin.com/api/v1/contracts/active", timeout=10)
        data = r.json()
        if data.get('code') == '200000':
            for item in data['data']:
                symbol = item['symbol']
                # KuCoin typically uses 8h funding intervals
                intervals[symbol] = "8h"
    except Exception as e:
        print(f"⚠️ KuCoin intervals error: {e}")
    return intervals


def fetch_raw_kucoin():
    """
    Fetch KuCoin funding rates using parallel requests for funding data
    """
    rates = {}
    try:
        intervals_map = get_kucoin_intervals_map()
        
        # Get all active contracts
        r = requests.get("https://api-futures.kucoin.com/api/v1/contracts/active", timeout=10)
        data = r.json()
        
        if data.get('code') == '200000':
            # Filter only USDTM contracts
            symbols = [item['symbol'] for item in data['data'] if item['symbol'].endswith('USDTM')]
            
            def fetch_funding(symbol):
                try:
                    ticker_r = requests.get(
                        f"https://api-futures.kucoin.com/api/v1/funding-rate/{symbol}/current",
                        timeout=5
                    )
                    ticker_data = ticker_r.json()
                    
                    if ticker_data.get('code') == '200000' and ticker_data.get('data'):
                        coin = symbol.replace('USDTM', '')
                        rate = float(ticker_data['data']['value']) * 100
                        interval = intervals_map.get(symbol, '8h')
                        return coin, (rate, interval)
                except:
                    pass
                return None, None
            
            # Use ThreadPoolExecutor for parallel requests
            with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
                futures = [executor.submit(fetch_funding, symbol) for symbol in symbols]
                for future in concurrent.futures.as_completed(futures):
                    coin, data = future.result()
                    if coin and data:
                        rates[coin] = data
                        
    except Exception as e:
        print(f"⚠️ KuCoin error: {e}")
    return rates



#
#
# --- AUTO-DETECTION LOGIC ---
AUTO_EXCHANGES = []
EXCHANGE_FUNCTIONS = {}

for name, obj in list(globals().items()):
    if name.startswith('fetch_raw_') and callable(obj):
        ex_name = name.replace('fetch_raw_', '')
        AUTO_EXCHANGES.append(ex_name)
        EXCHANGE_FUNCTIONS[ex_name] = obj

AUTO_EXCHANGES.sort()
print(f"🔍 AUTO-DETECTION: Found {len(AUTO_EXCHANGES)} exchanges: {AUTO_EXCHANGES}")

# --- MASTER FETCHER ---
def fetch_funding_data(exchange_list=None):
    if exchange_list is None:
        exchange_list = AUTO_EXCHANGES
        
    data = {}
    print(f"🚀 ENGINE STARTING: Fetching {exchange_list}...")
    
    def worker(ex_name):
        start = time.time()
        result = {}
        
        fetch_func = EXCHANGE_FUNCTIONS.get(ex_name)
        if fetch_func:
            result = fetch_func()
        else:
            print(f"⚠️ No function found for: {ex_name}")
            
        elapsed = time.time() - start
        print(f"   ✅ {ex_name}: {len(result)} pairs ({elapsed:.2f}s)")
        return ex_name, result

    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
        futures = [executor.submit(worker, ex) for ex in exchange_list]
        for future in concurrent.futures.as_completed(futures):
            name, res = future.result()
            data[name] = res
            
    return data

fetched_data = fetch_funding_data()
print("🏁 ENGINE COMPLETE. Data stored in variable 'fetched_data'.")


# CELL 3: THE UI (DASHBOARD LOGIC) & # CELL 4: EXECUTION & # CELL: TIME TABLE  # CELL: Dashboard Cards
# ═══════════════════════════════════════════════════════════════════════════════

def get_interval_color(interval, dark_mode=False):
    """Returns the color for funding interval based on timeline colors"""
    if not interval:
        return '#888888'
    
    interval_lower = interval.lower()
    
    if dark_mode:
        if '1h' in interval_lower:
            return '#4dabf7'  # Blue
        elif '4h' in interval_lower:
            return '#51cf66'  # Green
        elif '8h' in interval_lower:
            return '#ffd43b'  # Yellow
    else:
        if '1h' in interval_lower:
            return '#1976d2'  # Blue
        elif '4h' in interval_lower:
            return '#2e7d32'  # Green
        elif '8h' in interval_lower:
            return '#f57c00'  # Orange
    
    return '#888888'  # Default gray

def get_trade_link(exchange, coin):
    ex = exchange.lower()
    if ex == 'binance': return f"https://www.binance.com/en/futures/{coin}USDT"
    elif ex == 'bybit': return f"https://www.bybit.com/trade/usdt/{coin}USDT"
    elif ex == 'okx': return f"https://www.okx.com/trade-swap/{coin}-USDT-SWAP"
    elif ex == 'gate': return f"https://www.gate.io/futures_trade/{coin}_USDT"
    elif ex == 'bitget': return f"https://www.bitget.com/futures/{coin}USDT"
    elif ex == 'kucoin': return f"https://www.kucoin.com/trade/futures/{coin}USDTM"
    else: return "#"

def build_funding_dashboard(exchange_list, all_exchange_data, min_spread_percent=None, show_top=None, dark_mode=True):
    """
    Changed: Takes 'all_exchange_data' directly instead of a provider function.
    """
    
    html_template = Template("""
    <style>
        .funding-dashboard { border-collapse: collapse; width: auto; font-family: 'Consolas', monospace; font-size: 12px; background-color: {{ bg_main }}; }
        .funding-dashboard th { background-color: {{ bg_header }}; color: {{ text_header }}; padding: 10px 15px; position: sticky; top: 0; z-index: 10; border: 1px solid {{ border_color }}; text-align: center; }
        .funding-dashboard td { padding: 8px 15px; text-align: center; vertical-align: middle; border: 1px solid {{ border_color }}; }
        .funding-dashboard tr:nth-child(odd) { background-color: {{ bg_row_odd }}; }
        .funding-dashboard tr:nth-child(even) { background-color: {{ bg_row_even }}; }
        
        .coin-cell { font-weight: 700; color: {{ text_main }}; }
        .spread-cell { font-weight: 700; color: {{ text_main }}; }
        .info-cell { color: {{ text_main }}; font-size: 11px; }
        .exchange-count { background-color: {{ count_badge_color }}; border-radius: 10px; padding: 2px 8px; font-size: 10px; }
        .detail-sub { font-size: 10px; opacity: 0.8; margin-left: 4px; }
        
        a.rate-link { color: inherit; text-decoration: none; transition: color 0.2s; }
        a.rate-link:hover { color: #4dabf7 !important; text-decoration: underline; }
    </style>
    
    <div style="max-height: 800px; overflow-y: auto; border: 2px solid {{ border_color }}; border-radius: 8px;">
    <table class="funding-dashboard">
        <thead><tr>{% for col in columns %}<th>{{ col }}</th>{% endfor %}</tr></thead>
        <tbody>
        {% for row in rows %}
            <tr>
            {% for col in columns %}
                {% set cell = row[col] %}
                {% if cell.type == 'coin' %}<td class="coin-cell">{{ cell.value }}</td>
                {% elif cell.type == 'spread' %}<td class="spread-cell">{{ cell.value }}</td>
                {% elif cell.type == 'info_long' %}<td class="info-cell" style="color: {{ cell.color }};">{{ cell.value|safe }}</td>
                {% elif cell.type == 'info_short' %}<td class="info-cell" style="color: {{ cell.color }};">{{ cell.value|safe }}</td>
                {% elif cell.type == 'count' %}<td class="info-cell"><span class="exchange-count">{{ cell.value }}</span></td>
                {% elif cell.type == 'rate' %}
                    <td style="background-color:{{ cell.bg }}; color:{{ cell.color }}; font-weight:{{ cell.weight }}; font-size:{{ cell.size }}; text-align:center; vertical-align:middle;">
                        {{ cell.value|safe }}
                    </td>
                {% endif %}
            {% endfor %}
            </tr>
        {% endfor %}
        </tbody>
    </table>
    </div>
    """)

    # Data Processing (Uses 'all_exchange_data' directly)
    sri_lanka_tz = timezone(timedelta(hours=5, minutes=30))
   # print(f"\n⏰ {datetime.now(sri_lanka_tz).strftime('%Y-%m-%d %I:%M:%S')}")
    print(f"\n⏰ {datetime.now(sri_lanka_tz).strftime('%I:%M:%S')}")
    
    # Note: We use 'all_exchange_data' instead of calling a function here
    all_exchange_data = all_exchange_data 
    
    all_coins = set()
    for ex_data in all_exchange_data.values():
        all_coins.update(ex_data.keys())

    table_data = []
    for coin in sorted(all_coins):
        row = {'Coin': coin}
        rates_float_only = [] 
        exchanges_with_coin_info = [] 

        for exchange_id in exchange_list:
            ex_data_map = all_exchange_data.get(exchange_id, {})
            data_point = ex_data_map.get(coin)
            if data_point is not None:
                rate, interval = data_point
                row[exchange_id] = (rate, interval) 
                rates_float_only.append(rate)
                exchanges_with_coin_info.append((exchange_id, rate, interval))
            else:
                row[exchange_id] = None

        if len(rates_float_only) >= 2:
            max_rate = max(rates_float_only)
            min_rate = min(rates_float_only)
            row['Max Spread'] = max_rate - min_rate
            row['Long On'] = min(exchanges_with_coin_info, key=lambda x: x[1])
            row['Short On'] = max(exchanges_with_coin_info, key=lambda x: x[1])
            row['Exchanges'] = len(rates_float_only)
        else:
            row['Max Spread'] = None
            row['Long On'] = None
            row['Short On'] = None
            row['Exchanges'] = len(rates_float_only)
        table_data.append(row)

    df = pd.DataFrame(table_data)
    df = df.dropna(subset=['Max Spread'])
    if min_spread_percent is not None:
        df = df[df['Max Spread'] >= min_spread_percent]
    df = df.sort_values('Max Spread', ascending=False)
    
    fixed_cols = ['Coin', 'Max Spread', 'Long On', 'Short On', 'Exchanges']
    valid_ex_cols = [c for c in exchange_list if c in df.columns]
    df = df[fixed_cols + valid_ex_cols].reset_index(drop=True)
    
    if show_top: df = df.head(show_top)
    if len(df) == 0:
        print("❌ No opportunities found.")
        return

    # Render Preparation
    rows_for_template = []
    for _, row in df.iterrows():
        current_row_dict = {}
        row_floats = [row[c][0] for c in valid_ex_cols if row[c] is not None]
        row_min = min(row_floats) if row_floats else None
        row_max = max(row_floats) if row_floats else None

        for col in df.columns:
            val = row[col]
            if col == 'Coin': current_row_dict[col] = {'type': 'coin', 'value': val}
            elif col == 'Max Spread': current_row_dict[col] = {'type': 'spread', 'value': f"{val:.4f}%"}
            elif col == 'Long On':
                ex_name, ex_rate, ex_inv = val
                color = "#51cf66" if dark_mode else "#2e7d32"
                url = get_trade_link(ex_name, row['Coin'])
                interval_color = get_interval_color(ex_inv, dark_mode)
                disp = f'<a href="{url}" target="_blank" class="rate-link" style="color: {color};">{ex_name} <span class="detail-sub">({ex_rate:.4f}%<sup style="color:{interval_color}; font-weight:600;">{ex_inv}</sup>)</span></a>'
                current_row_dict[col] = {'type': 'info_long', 'color': color, 'value': disp}
            elif col == 'Short On':
                ex_name, ex_rate, ex_inv = val
                color = "#ff6b6b" if dark_mode else "#c62828"
                url = get_trade_link(ex_name, row['Coin'])
                interval_color = get_interval_color(ex_inv, dark_mode)
                disp = f'<a href="{url}" target="_blank" class="rate-link" style="color: {color};">{ex_name} <span class="detail-sub">({ex_rate:.4f}%<sup style="color:{interval_color}; font-weight:600;">{ex_inv}</sup>)</span></a>'
                current_row_dict[col] = {'type': 'info_short', 'color': color, 'value': disp}
            elif col == 'Exchanges': current_row_dict[col] = {'type': 'count', 'value': int(val)}
            elif col in valid_ex_cols:
                bg, txt, disp = get_funding_color_dynamic(val, row_floats, dark_mode=dark_mode)
                url = get_trade_link(col, row['Coin'])
                disp = f'<a href="{url}" target="_blank" class="rate-link" style="color: inherit;">{disp}</a>'
                
                is_best = False
                if val is not None and val[0] in [row_min, row_max]:
                    is_best = True
                weight = "900" if is_best else "400"
                size = "13px" if is_best else "12px" 
                if is_best: disp = disp.replace('</a>', '<sup style="margin-left:1px;">*</sup></a>')
                
                current_row_dict[col] = {'type': 'rate', 'bg': bg, 'color': txt, 'weight': weight, 'size': size, 'value': disp}
        rows_for_template.append(current_row_dict)

    # Final Render
    if dark_mode:
        bg_main, bg_header, border_color = '#1a1a1a', '#0d1117', '#30363d'
        bg_row_even, bg_row_odd, text_main, text_header = '#1e1e1e', '#242424', '#e6e6e6', '#ffffff'
        count_badge_color = '#3d3d3d'
    else:
        bg_main, bg_header, border_color = '#ffffff', '#1a252f', '#e0e0e0'
        bg_row_even, bg_row_odd, text_main, text_header = '#f8f9fa', '#ffffff', '#2c3e50', '#ffffff'
        count_badge_color = '#e9ecef'

    html_output = html_template.render(
        columns=df.columns, rows=rows_for_template,
        bg_main=bg_main, bg_header=bg_header, border_color=border_color,
        bg_row_even=bg_row_even, bg_row_odd=bg_row_odd,
        text_main=text_main, text_header=text_header, count_badge_color=count_badge_color
    )
    spread_val = min_spread_percent if min_spread_percent is not None else 0
    print(f"🏆 Top {len(df)} Opps Spread >{spread_val}% ")
    display(HTML(html_output))

print("✅ UI Logic loaded.")
print()

# CELL 4: EXECUTION
# ═══════════════════════════════════════════════════════════════════════════════
# We use the variable 'fetched_data' created in Cell 2
# You can check if data exists by running: print(fetched_data.keys())

# USAGE 

SCALE      = 1.0
DARK_MODE  = False
MIN_SPREAD = 0.4

build_funding_dashboard(exchange_list=AUTO_EXCHANGES, all_exchange_data=fetched_data, 
     min_spread_percent=MIN_SPREAD, show_top=20,
     dark_mode=DARK_MODE)

🔍 AUTO-DETECTION: Found 3 exchanges: ['binance', 'bybit', 'kucoin']
🚀 ENGINE STARTING: Fetching ['binance', 'bybit', 'kucoin']...
   ✅ binance: 671 pairs (3.54s)
   ✅ bybit: 537 pairs (4.32s)
   ✅ kucoin: 523 pairs (102.48s)
🏁 ENGINE COMPLETE. Data stored in variable 'fetched_data'.
✅ UI Logic loaded.


⏰ 07:45:34
🏆 Top 7 Opps Spread >0.4% 


Coin,Max Spread,Long On,Short On,Exchanges,binance,bybit,kucoin
AXL,0.6699%,kucoin (-1.8111%8h),binance (-1.1412%4h),3,-1.1412%4h*,-1.2961%4h,-1.8111%8h*
ENJ,0.5386%,kucoin (-0.6366%8h),binance (-0.0980%1h),3,-0.0980%1h*,-0.5600%8h,-0.6366%8h*
SIREN,0.4379%,bybit (-0.5671%4h),binance (-0.1292%1h),3,-0.1292%1h*,-0.5671%4h*,-0.3400%8h
XCN,0.4364%,bybit (-0.4364%8h),binance (0.0000%8h),2,0.0000%8h*,-0.4364%8h*,-
BSU,0.4112%,kucoin (-0.1467%8h),bybit (0.2645%4h),2,-,0.2645%4h*,-0.1467%8h*
MBOX,0.4069%,kucoin (-0.4716%8h),binance (-0.0647%4h),3,-0.0647%4h*,-0.1534%4h,-0.4716%8h*
OXT,0.4069%,binance (0.0000%8h),kucoin (0.4069%8h),3,0.0000%8h*,0.0100%8h,0.4069%8h*


TEST1

In [3]:

# CELL 2: THE ENGINE (AUTO-DETECTION ENABLED)
# ═══════════════════════════════════════════════════════════════════════════════

def get_binance_intervals_map():
    """Fetch funding intervals from Binance fundingInfo endpoint"""
    intervals = {}
    try:
        r = requests.get("https://fapi.binance.com/fapi/v1/fundingInfo", timeout=10)
        data = r.json()
        for item in data:
            symbol = item['symbol']
            hours = item.get('fundingIntervalHours', 8)
            intervals[symbol] = f"{hours}h"
    except Exception as e:
        print(f"⚠️ Binance intervals error: {e}")
    return intervals


def fetch_raw_binance():
    """
    Uses lastFundingRate from API - this is the accurate settled rate.
    Manual calculation is inaccurate because Binance uses TWAP over 8 hours.
    """
    rates = {}
    try:
        intervals_map = get_binance_intervals_map()
        r = requests.get("https://fapi.binance.com/fapi/v1/premiumIndex", timeout=10)
        
        for item in r.json():
            symbol = item['symbol']
            if symbol.endswith('USDT') and 'USDC' not in symbol:
                coin = symbol.replace('USDT', '')
                
                # ✅ Use API-provided rate directly (accurate!)
                rate = float(item['lastFundingRate']) * 100
                
                interval = intervals_map.get(symbol, '8h')
                rates[coin] = (rate, interval)
                
    except Exception as e:
        print(f"⚠️ Binance error: {e}")
    return rates


def get_bybit_intervals_map():
    """Fetch funding intervals from Bybit instruments endpoint"""
    intervals = {}
    try:
        r = requests.get("https://api.bybit.com/v5/market/instruments-info?category=linear", timeout=10)
        data = r.json()
        if data.get('retCode') == 0:
            for item in data['result']['list']:
                funding_interval = item.get('fundingInterval', 480)
                
                if isinstance(funding_interval, int):
                    h = funding_interval // 60
                elif isinstance(funding_interval, str):
                    h = int(funding_interval.split(':')[0])
                else:
                    h = 8
                    
                intervals[item['symbol']] = f"{h}h"
    except Exception as e:
        print(f"⚠️ Bybit intervals error: {e}")
    return intervals


def fetch_raw_bybit():
    """
    Bybit fundingRate is the predicted rate for next settlement.
    """
    rates = {}
    try:
        intervals_map = get_bybit_intervals_map()
        r = requests.get("https://api.bybit.com/v5/market/tickers?category=linear", timeout=10)
        data = r.json()
        if data.get('retCode') == 0:
            for item in data['result']['list']:
                if item['symbol'].endswith('USDT'):
                    coin = item['symbol'].replace('USDT', '')
                    # ✅ API-provided predicted rate
                    rate = float(item['fundingRate']) * 100
                    interval = intervals_map.get(item['symbol'], '8h')
                    rates[coin] = (rate, interval)
    except Exception as e:
        print(f"⚠️ Bybit error: {e}")
    return rates


def get_kucoin_intervals_map():
    """Fetch funding intervals from KuCoin contracts endpoint"""
    intervals = {}
    try:
        r = requests.get("https://api-futures.kucoin.com/api/v1/contracts/active", timeout=10)
        data = r.json()
        if data.get('code') == '200000':
            for item in data['data']:
                symbol = item['symbol']
                funding_symbol = item.get('fundingRateSymbol', '')
                if '4H' in funding_symbol:
                    intervals[symbol] = '4h'
                elif '1H' in funding_symbol:
                    intervals[symbol] = '1h'
                else:
                    intervals[symbol] = '8h'
    except Exception as e:
        print(f"⚠️ KuCoin intervals error: {e}")
    return intervals


def fetch_raw_kucoin():
    """
    Uses fundingFeeRate - the current rate to be settled next.
    """
    rates = {}
    try:
        r = requests.get("https://api-futures.kucoin.com/api/v1/contracts/active", timeout=10)
        data = r.json()
        
        if data.get('code') == '200000':
            for item in data['data']:
                symbol = item['symbol']
                
                if symbol.endswith('USDTM'):
                    coin = symbol.replace('USDTM', '')
                    if coin == 'XBT':
                        coin = 'BTC'
                    
                    # ✅ Use fundingFeeRate (current rate to be settled)
                    rate = float(item.get('fundingFeeRate', 0)) * 100
                    
                    funding_symbol = item.get('fundingRateSymbol', '')
                    if '4H' in funding_symbol:
                        interval = '4h'
                    elif '1H' in funding_symbol:
                        interval = '1h'
                    else:
                        interval = '8h'
                    
                    rates[coin] = (rate, interval)
                    
    except Exception as e:
        print(f"⚠️ KuCoin error: {e}")
    return rates

# --- AUTO-DETECTION LOGIC ---
AUTO_EXCHANGES = []
EXCHANGE_FUNCTIONS = {}

for name, obj in list(globals().items()):
    if name.startswith('fetch_raw_') and callable(obj):
        ex_name = name.replace('fetch_raw_', '')
        AUTO_EXCHANGES.append(ex_name)
        EXCHANGE_FUNCTIONS[ex_name] = obj

AUTO_EXCHANGES.sort()
print(f"🔍 AUTO-DETECTION: Found {len(AUTO_EXCHANGES)} exchanges: {AUTO_EXCHANGES}")

# --- MASTER FETCHER ---
def fetch_funding_data(exchange_list=None):
    if exchange_list is None:
        exchange_list = AUTO_EXCHANGES
        
    data = {}
    print(f"🚀 ENGINE STARTING: Fetching {exchange_list}...")
    
    def worker(ex_name):
        start = time.time()
        result = {}
        
        fetch_func = EXCHANGE_FUNCTIONS.get(ex_name)
        if fetch_func:
            result = fetch_func()
        else:
            print(f"⚠️ No function found for: {ex_name}")
            
        elapsed = time.time() - start
        print(f"   ✅ {ex_name}: {len(result)} pairs ({elapsed:.2f}s)")
        return ex_name, result

    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
        futures = [executor.submit(worker, ex) for ex in exchange_list]
        for future in concurrent.futures.as_completed(futures):
            name, res = future.result()
            data[name] = res
            
    return data

fetched_data = fetch_funding_data()
print("🏁 ENGINE COMPLETE. Data stored in variable 'fetched_data'.")


# CELL 3: THE UI (DASHBOARD LOGIC) & # CELL 4: EXECUTION & # CELL: TIME TABLE  # CELL: Dashboard Cards
# ═══════════════════════════════════════════════════════════════════════════════

def get_interval_color(interval, dark_mode=False):
    """Returns the color for funding interval based on timeline colors"""
    if not interval:
        return '#888888'
    
    interval_lower = interval.lower()
    
    if dark_mode:
        if '1h' in interval_lower:
            return '#4dabf7'  # Blue
        elif '4h' in interval_lower:
            return '#51cf66'  # Green
        elif '8h' in interval_lower:
            return '#ffd43b'  # Yellow
    else:
        if '1h' in interval_lower:
            return '#1976d2'  # Blue
        elif '4h' in interval_lower:
            return '#2e7d32'  # Green
        elif '8h' in interval_lower:
            return '#f57c00'  # Orange
    
    return '#888888'  # Default gray

def get_trade_link(exchange, coin):
    ex = exchange.lower()
    if ex == 'binance': return f"https://www.binance.com/en/futures/{coin}USDT"
    elif ex == 'bybit': return f"https://www.bybit.com/trade/usdt/{coin}USDT"
    elif ex == 'okx': return f"https://www.okx.com/trade-swap/{coin}-USDT-SWAP"
    elif ex == 'gate': return f"https://www.gate.io/futures_trade/{coin}_USDT"
    elif ex == 'bitget': return f"https://www.bitget.com/futures/{coin}USDT"
    elif ex == 'kucoin': return f"https://www.kucoin.com/trade/futures/{coin}USDTM"
    else: return "#"

def build_funding_dashboard(exchange_list, all_exchange_data, min_spread_percent=None, show_top=None, dark_mode=True):
    """
    Changed: Takes 'all_exchange_data' directly instead of a provider function.
    """
    
    html_template = Template("""
    <style>
        .funding-dashboard { border-collapse: collapse; width: auto; font-family: 'Consolas', monospace; font-size: 12px; background-color: {{ bg_main }}; }
        .funding-dashboard th { background-color: {{ bg_header }}; color: {{ text_header }}; padding: 10px 15px; position: sticky; top: 0; z-index: 10; border: 1px solid {{ border_color }}; text-align: center; }
        .funding-dashboard td { padding: 8px 15px; text-align: center; vertical-align: middle; border: 1px solid {{ border_color }}; }
        .funding-dashboard tr:nth-child(odd) { background-color: {{ bg_row_odd }}; }
        .funding-dashboard tr:nth-child(even) { background-color: {{ bg_row_even }}; }
        
        .coin-cell { font-weight: 700; color: {{ text_main }}; }
        .spread-cell { font-weight: 700; color: {{ text_main }}; }
        .info-cell { color: {{ text_main }}; font-size: 11px; }
        .exchange-count { background-color: {{ count_badge_color }}; border-radius: 10px; padding: 2px 8px; font-size: 10px; }
        .detail-sub { font-size: 10px; opacity: 0.8; margin-left: 4px; }
        
        a.rate-link { color: inherit; text-decoration: none; transition: color 0.2s; }
        a.rate-link:hover { color: #4dabf7 !important; text-decoration: underline; }
    </style>
    
    <div style="max-height: 800px; overflow-y: auto; border: 2px solid {{ border_color }}; border-radius: 8px;">
    <table class="funding-dashboard">
        <thead><tr>{% for col in columns %}<th>{{ col }}</th>{% endfor %}</tr></thead>
        <tbody>
        {% for row in rows %}
            <tr>
            {% for col in columns %}
                {% set cell = row[col] %}
                {% if cell.type == 'coin' %}<td class="coin-cell">{{ cell.value }}</td>
                {% elif cell.type == 'spread' %}<td class="spread-cell">{{ cell.value }}</td>
                {% elif cell.type == 'info_long' %}<td class="info-cell" style="color: {{ cell.color }};">{{ cell.value|safe }}</td>
                {% elif cell.type == 'info_short' %}<td class="info-cell" style="color: {{ cell.color }};">{{ cell.value|safe }}</td>
                {% elif cell.type == 'count' %}<td class="info-cell"><span class="exchange-count">{{ cell.value }}</span></td>
                {% elif cell.type == 'rate' %}
                    <td style="background-color:{{ cell.bg }}; color:{{ cell.color }}; font-weight:{{ cell.weight }}; font-size:{{ cell.size }}; text-align:center; vertical-align:middle;">
                        {{ cell.value|safe }}
                    </td>
                {% endif %}
            {% endfor %}
            </tr>
        {% endfor %}
        </tbody>
    </table>
    </div>
    """)

    # Data Processing (Uses 'all_exchange_data' directly)
    sri_lanka_tz = timezone(timedelta(hours=5, minutes=30))
   # print(f"\n⏰ {datetime.now(sri_lanka_tz).strftime('%Y-%m-%d %I:%M:%S')}")
    print(f"\n⏰ {datetime.now(sri_lanka_tz).strftime('%I:%M:%S')}")
    
    # Note: We use 'all_exchange_data' instead of calling a function here
    all_exchange_data = all_exchange_data 
    
    all_coins = set()
    for ex_data in all_exchange_data.values():
        all_coins.update(ex_data.keys())

    table_data = []
    for coin in sorted(all_coins):
        row = {'Coin': coin}
        rates_float_only = [] 
        exchanges_with_coin_info = [] 

        for exchange_id in exchange_list:
            ex_data_map = all_exchange_data.get(exchange_id, {})
            data_point = ex_data_map.get(coin)
            if data_point is not None:
                rate, interval = data_point
                row[exchange_id] = (rate, interval) 
                rates_float_only.append(rate)
                exchanges_with_coin_info.append((exchange_id, rate, interval))
            else:
                row[exchange_id] = None

        if len(rates_float_only) >= 2:
            max_rate = max(rates_float_only)
            min_rate = min(rates_float_only)
            row['Max Spread'] = max_rate - min_rate
            row['Long On'] = min(exchanges_with_coin_info, key=lambda x: x[1])
            row['Short On'] = max(exchanges_with_coin_info, key=lambda x: x[1])
            row['Exchanges'] = len(rates_float_only)
        else:
            row['Max Spread'] = None
            row['Long On'] = None
            row['Short On'] = None
            row['Exchanges'] = len(rates_float_only)
        table_data.append(row)

    df = pd.DataFrame(table_data)
    df = df.dropna(subset=['Max Spread'])
    if min_spread_percent is not None:
        df = df[df['Max Spread'] >= min_spread_percent]
    df = df.sort_values('Max Spread', ascending=False)
    
    fixed_cols = ['Coin', 'Max Spread', 'Long On', 'Short On', 'Exchanges']
    valid_ex_cols = [c for c in exchange_list if c in df.columns]
    df = df[fixed_cols + valid_ex_cols].reset_index(drop=True)
    
    if show_top: df = df.head(show_top)
    if len(df) == 0:
        print("❌ No opportunities found.")
        return

    # Render Preparation
    rows_for_template = []
    for _, row in df.iterrows():
        current_row_dict = {}
        row_floats = [row[c][0] for c in valid_ex_cols if row[c] is not None]
        row_min = min(row_floats) if row_floats else None
        row_max = max(row_floats) if row_floats else None

        for col in df.columns:
            val = row[col]
            if col == 'Coin': current_row_dict[col] = {'type': 'coin', 'value': val}
            elif col == 'Max Spread': current_row_dict[col] = {'type': 'spread', 'value': f"{val:.4f}%"}
            elif col == 'Long On':
                ex_name, ex_rate, ex_inv = val
                color = "#51cf66" if dark_mode else "#2e7d32"
                url = get_trade_link(ex_name, row['Coin'])
                interval_color = get_interval_color(ex_inv, dark_mode)
                disp = f'<a href="{url}" target="_blank" class="rate-link" style="color: {color};">{ex_name} <span class="detail-sub">({ex_rate:.4f}%<sup style="color:{interval_color}; font-weight:600;">{ex_inv}</sup>)</span></a>'
                current_row_dict[col] = {'type': 'info_long', 'color': color, 'value': disp}
            elif col == 'Short On':
                ex_name, ex_rate, ex_inv = val
                color = "#ff6b6b" if dark_mode else "#c62828"
                url = get_trade_link(ex_name, row['Coin'])
                interval_color = get_interval_color(ex_inv, dark_mode)
                disp = f'<a href="{url}" target="_blank" class="rate-link" style="color: {color};">{ex_name} <span class="detail-sub">({ex_rate:.4f}%<sup style="color:{interval_color}; font-weight:600;">{ex_inv}</sup>)</span></a>'
                current_row_dict[col] = {'type': 'info_short', 'color': color, 'value': disp}
            elif col == 'Exchanges': current_row_dict[col] = {'type': 'count', 'value': int(val)}
            elif col in valid_ex_cols:
                bg, txt, disp = get_funding_color_dynamic(val, row_floats, dark_mode=dark_mode)
                url = get_trade_link(col, row['Coin'])
                disp = f'<a href="{url}" target="_blank" class="rate-link" style="color: inherit;">{disp}</a>'
                
                is_best = False
                if val is not None and val[0] in [row_min, row_max]:
                    is_best = True
                weight = "900" if is_best else "400"
                size = "13px" if is_best else "12px" 
                if is_best: disp = disp.replace('</a>', '<sup style="margin-left:1px;">*</sup></a>')
                
                current_row_dict[col] = {'type': 'rate', 'bg': bg, 'color': txt, 'weight': weight, 'size': size, 'value': disp}
        rows_for_template.append(current_row_dict)

    # Final Render
    if dark_mode:
        bg_main, bg_header, border_color = '#1a1a1a', '#0d1117', '#30363d'
        bg_row_even, bg_row_odd, text_main, text_header = '#1e1e1e', '#242424', '#e6e6e6', '#ffffff'
        count_badge_color = '#3d3d3d'
    else:
        bg_main, bg_header, border_color = '#ffffff', '#1a252f', '#e0e0e0'
        bg_row_even, bg_row_odd, text_main, text_header = '#f8f9fa', '#ffffff', '#2c3e50', '#ffffff'
        count_badge_color = '#e9ecef'

    html_output = html_template.render(
        columns=df.columns, rows=rows_for_template,
        bg_main=bg_main, bg_header=bg_header, border_color=border_color,
        bg_row_even=bg_row_even, bg_row_odd=bg_row_odd,
        text_main=text_main, text_header=text_header, count_badge_color=count_badge_color
    )
    spread_val = min_spread_percent if min_spread_percent is not None else 0
    print(f"🏆 Top {len(df)} Opps Spread >{spread_val}% ")
    display(HTML(html_output))

print("✅ UI Logic loaded.")
print()

# CELL 4: EXECUTION
# ═══════════════════════════════════════════════════════════════════════════════
# We use the variable 'fetched_data' created in Cell 2
# You can check if data exists by running: print(fetched_data.keys())

# USAGE 

SCALE      = 1.0
DARK_MODE  = False
MIN_SPREAD = 0.4

build_funding_dashboard(exchange_list=AUTO_EXCHANGES, all_exchange_data=fetched_data, 
     min_spread_percent=MIN_SPREAD, show_top=20,
     dark_mode=DARK_MODE)

🔍 AUTO-DETECTION: Found 3 exchanges: ['binance', 'bybit', 'kucoin']
🚀 ENGINE STARTING: Fetching ['binance', 'bybit', 'kucoin']...
   ✅ kucoin: 563 pairs (1.82s)
   ✅ binance: 671 pairs (2.12s)
   ✅ bybit: 537 pairs (2.66s)
🏁 ENGINE COMPLETE. Data stored in variable 'fetched_data'.
✅ UI Logic loaded.


⏰ 07:48:03
🏆 Top 7 Opps Spread >0.4% 


Coin,Max Spread,Long On,Short On,Exchanges,binance,bybit,kucoin
ENJ,0.7888%,kucoin (-0.9909%8h),binance (-0.2021%1h),3,-0.2021%1h*,-0.7887%8h,-0.9909%8h*
AXL,0.7145%,kucoin (-1.8170%8h),binance (-1.1025%4h),3,-1.1025%4h*,-1.2440%4h,-1.8170%8h*
BSU,0.4371%,kucoin (-0.1500%8h),bybit (0.2871%4h),2,-,0.2871%4h*,-0.1500%8h*
SIREN,0.4292%,bybit (-0.5546%4h),binance (-0.1255%1h),3,-0.1255%1h*,-0.5546%4h*,-0.3336%8h
XCN,0.4283%,bybit (-0.4283%8h),binance (0.0000%8h),2,0.0000%8h*,-0.4283%8h*,-
MBOX,0.4051%,kucoin (-0.4505%8h),binance (-0.0454%4h),3,-0.0454%4h*,-0.1529%4h,-0.4505%8h*
OXT,0.4005%,binance (0.0000%8h),kucoin (0.4005%8h),3,0.0000%8h*,0.0100%8h,0.4005%8h*


TEST2

In [4]:
# CELL 4: THE UI (DASHBOARD LOGIC)
# ═══════════════════════════════════════════════════════════════════════════════

def get_interval_color(interval, dark_mode=False):
    """Returns the color for funding interval based on timeline colors"""
    if not interval:
        return '#888888'
    interval_lower = interval.lower()
    if dark_mode:
        if '1h' in interval_lower: return '#4dabf7'
        elif '4h' in interval_lower: return '#51cf66'
        elif '8h' in interval_lower: return '#ffd43b'
    else:
        if '1h' in interval_lower: return '#1976d2'
        elif '4h' in interval_lower: return '#2e7d32'
        elif '8h' in interval_lower: return '#f57c00'
    return '#888888'

def get_trade_link(exchange, coin):
    ex = exchange.lower()
    if ex == 'binance': return f"https://www.binance.com/en/futures/{coin}USDT"
    elif ex == 'bybit': return f"https://www.bybit.com/trade/usdt/{coin}USDT"
    elif ex == 'okx': return f"https://www.okx.com/trade-swap/{coin}-USDT-SWAP"
    elif ex == 'gate': return f"https://www.gate.io/futures_trade/{coin}_USDT"
    elif ex == 'bitget': return f"https://www.bitget.com/futures/{coin}USDT"
    elif ex == 'kucoin': return f"https://www.kucoin.com/trade/futures/{coin}USDTM"
    else: return "#"

def build_funding_dashboard(exchange_list, all_exchange_data, min_imbalance=None, 
                            show_top=None, dark_mode=True):
    """
    Dashboard focused on Interval Imbalance.
    
    min_imbalance: minimum funding rate VALUE (%) on non-8h exchanges to show
                   e.g., 0.02 means only show rows where a non-8h exchange has rate >= 0.02%
    """
    
    html_template = Template("""
    <style>
        .funding-dashboard { border-collapse: collapse; width: auto; font-family: 'Consolas', monospace; font-size: 12px; background-color: {{ bg_main }}; }
        .funding-dashboard th { background-color: {{ bg_header }}; color: {{ text_header }}; padding: 10px 15px; position: sticky; top: 0; z-index: 10; border: 1px solid {{ border_color }}; text-align: center; }
        .funding-dashboard td { padding: 8px 15px; text-align: center; vertical-align: middle; border: 1px solid {{ border_color }}; }
        .funding-dashboard tr:nth-child(odd) { background-color: {{ bg_row_odd }}; }
        .funding-dashboard tr:nth-child(even) { background-color: {{ bg_row_even }}; }
        
        .coin-cell { font-weight: 700; color: {{ text_main }}; }
        .info-cell { color: {{ text_main }}; font-size: 11px; }
        .exchange-count { background-color: {{ count_badge_color }}; border-radius: 10px; padding: 2px 8px; font-size: 10px; }
        
        .imbalance-cell { font-size: 11px; font-weight: 600; }
        .imbalance-tag { padding: 2px 6px; border-radius: 4px; margin: 1px 2px; display: inline-block; }
        .imbalance-1h { background-color: {{ imbalance_1h_bg }}; color: {{ imbalance_1h_text }}; }
        .imbalance-4h { background-color: {{ imbalance_4h_bg }}; color: {{ imbalance_4h_text }}; }
        
        a.rate-link { color: inherit; text-decoration: none; transition: color 0.2s; }
        a.rate-link:hover { color: #4dabf7 !important; text-decoration: underline; }
    </style>
    
    <div style="max-height: 800px; overflow-y: auto; border: 2px solid {{ border_color }}; border-radius: 8px;">
    <table class="funding-dashboard">
        <thead><tr>{% for col in columns %}<th>{{ col }}</th>{% endfor %}</tr></thead>
        <tbody>
        {% for row in rows %}
            <tr>
            {% for col in columns %}
                {% set cell = row[col] %}
                {% if cell.type == 'coin' %}<td class="coin-cell">{{ cell.value }}</td>
                {% elif cell.type == 'imbalance' %}<td class="imbalance-cell">{{ cell.value|safe }}</td>
                {% elif cell.type == 'count' %}<td class="info-cell"><span class="exchange-count">{{ cell.value }}</span></td>
                {% elif cell.type == 'rate' %}
                    <td style="background-color:{{ cell.bg }}; color:{{ cell.color }}; font-weight:{{ cell.weight }}; font-size:{{ cell.size }}; text-align:center; vertical-align:middle;">
                        {{ cell.value|safe }}
                    </td>
                {% endif %}
            {% endfor %}
            </tr>
        {% endfor %}
        </tbody>
    </table>
    </div>
    """)

    # Timezone display
    sri_lanka_tz = timezone(timedelta(hours=5, minutes=30))
    print(f"\n⏰ {datetime.now(sri_lanka_tz).strftime('%I:%M:%S')}")
    
    # Collect all coins
    all_coins = set()
    for ex_data in all_exchange_data.values():
        all_coins.update(ex_data.keys())

    # Build table data
    table_data = []
    for coin in sorted(all_coins):
        row = {'Coin': coin}
        rates_float_only = []
        all_intervals = []
        non_8h_exchanges = []  # Track non-8h intervals with their rates
        
        for exchange_id in exchange_list:
            ex_data_map = all_exchange_data.get(exchange_id, {})
            data_point = ex_data_map.get(coin)
            if data_point is not None:
                rate, interval = data_point
                row[exchange_id] = (rate, interval)
                rates_float_only.append(rate)
                all_intervals.append(interval.lower() if interval else '8h')
                
                # Track non-8h intervals with rate
                if interval and interval.lower() != '8h':
                    non_8h_exchanges.append((exchange_id, interval, rate))
            else:
                row[exchange_id] = None

        if len(rates_float_only) >= 2:
            row['Exchanges'] = len(rates_float_only)
            
            # Interval Imbalance: only show if there's a MIX of intervals (not all same)
            unique_intervals = set(all_intervals)
            if len(unique_intervals) > 1 and non_8h_exchanges:
                # There's a mix AND some are non-8h
                row['_non_8h_list'] = non_8h_exchanges
                # Imbalance value = max absolute rate among non-8h exchanges
                row['_imbalance_value'] = max(abs(r) for _, _, r in non_8h_exchanges)
            else:
                # All same interval OR no non-8h exchanges
                row['_non_8h_list'] = []
                row['_imbalance_value'] = 0
        else:
            row['Exchanges'] = len(rates_float_only)
            row['_non_8h_list'] = []
            row['_imbalance_value'] = 0
        
        table_data.append(row)

    # Create DataFrame and filter
    df = pd.DataFrame(table_data)
    df = df[df['Exchanges'] >= 2]  # Need at least 2 exchanges
    
    # Filter by imbalance VALUE (the rate on non-8h exchanges)
    if min_imbalance is not None:
        df = df[df['_imbalance_value'] >= min_imbalance]
    
    # Sort by imbalance value (highest first)
    df = df.sort_values('_imbalance_value', ascending=False)
    
    # Define columns
    fixed_cols = ['Coin', 'Interval Imbalance', 'Exchanges']
    valid_ex_cols = [c for c in exchange_list if c in df.columns]
    
    df = df.reset_index(drop=True)
    if show_top:
        df = df.head(show_top)
    
    if len(df) == 0:
        print("❌ No opportunities found matching filters.")
        return

    # Build template rows
    rows_for_template = []
    for _, row in df.iterrows():
        current_row_dict = {}
        row_floats = [row[c][0] for c in valid_ex_cols if row[c] is not None]
        row_min = min(row_floats) if row_floats else None
        row_max = max(row_floats) if row_floats else None

        for col in fixed_cols + valid_ex_cols:
            if col == 'Coin':
                current_row_dict[col] = {'type': 'coin', 'value': row[col]}
                
            elif col == 'Interval Imbalance':
                non_8h_list = row['_non_8h_list']
                if non_8h_list:
                    # Format as styled tags with exchange:interval(rate)
                    tags = []
                    for ex, inv, rate in non_8h_list:
                        inv_lower = inv.lower()
                        if '1h' in inv_lower:
                            css_class = 'imbalance-1h'
                        elif '4h' in inv_lower:
                            css_class = 'imbalance-4h'
                        else:
                            css_class = ''
                        sign = '+' if rate > 0 else ''
                        tags.append(f'<span class="imbalance-tag {css_class}">{ex}:{inv} ({sign}{rate:.4f}%)</span>')
                    display_val = ' '.join(tags)
                else:
                    display_val = '—'
                current_row_dict[col] = {'type': 'imbalance', 'value': display_val}
                
            elif col == 'Exchanges':
                current_row_dict[col] = {'type': 'count', 'value': int(row[col])}
                
            elif col in valid_ex_cols:
                val = row[col]
                if val is not None:
                    bg, txt, disp = get_funding_color_dynamic(val, row_floats, dark_mode=dark_mode)
                    url = get_trade_link(col, row['Coin'])
                    disp = f'<a href="{url}" target="_blank" class="rate-link" style="color: inherit;">{disp}</a>'
                    
                    is_best = val[0] in [row_min, row_max]
                    weight = "900" if is_best else "400"
                    size = "13px" if is_best else "12px"
                    if is_best:
                        disp = disp.replace('</a>', '<sup style="margin-left:1px;">*</sup></a>')
                    
                    current_row_dict[col] = {'type': 'rate', 'bg': bg, 'color': txt, 'weight': weight, 'size': size, 'value': disp}
                else:
                    current_row_dict[col] = {'type': 'rate', 'bg': 'transparent', 'color': '#555', 'weight': '400', 'size': '11px', 'value': '—'}
        
        rows_for_template.append(current_row_dict)

    # Theme colors
    if dark_mode:
        bg_main, bg_header, border_color = '#1a1a1a', '#0d1117', '#30363d'
        bg_row_even, bg_row_odd = '#1e1e1e', '#242424'
        text_main, text_header = '#e6e6e6', '#ffffff'
        count_badge_color = '#3d3d3d'
        imbalance_1h_bg, imbalance_1h_text = '#1a3a5c', '#4dabf7'
        imbalance_4h_bg, imbalance_4h_text = '#1a3d1a', '#51cf66'
    else:
        bg_main, bg_header, border_color = '#ffffff', '#1a252f', '#e0e0e0'
        bg_row_even, bg_row_odd = '#f8f9fa', '#ffffff'
        text_main, text_header = '#2c3e50', '#ffffff'
        count_badge_color = '#e9ecef'
        imbalance_1h_bg, imbalance_1h_text = '#e3f2fd', '#1565c0'
        imbalance_4h_bg, imbalance_4h_text = '#e8f5e9', '#2e7d32'

    html_output = html_template.render(
        columns=fixed_cols + valid_ex_cols, 
        rows=rows_for_template,
        bg_main=bg_main, bg_header=bg_header, border_color=border_color,
        bg_row_even=bg_row_even, bg_row_odd=bg_row_odd,
        text_main=text_main, text_header=text_header, 
        count_badge_color=count_badge_color,
        imbalance_1h_bg=imbalance_1h_bg, imbalance_1h_text=imbalance_1h_text,
        imbalance_4h_bg=imbalance_4h_bg, imbalance_4h_text=imbalance_4h_text
    )
    
    # Summary
    imb_val = min_imbalance if min_imbalance is not None else 0
    print(f"🏆 Top {len(df)} Opps | Imbalance Rate ≥ {imb_val}%")
    display(HTML(html_output))

print("✅ UI Logic loaded.")
print()

# CELL 4: EXECUTION
# ═══════════════════════════════════════════════════════════════════════════════

DARK_MODE     = False
MIN_IMBALANCE = 0.3    # Minimum funding rate VALUE (%) on non-8h exchanges
SHOW_TOP      = 30

build_funding_dashboard(
    exchange_list=AUTO_EXCHANGES, 
    all_exchange_data=fetched_data, 
    min_imbalance=MIN_IMBALANCE,    # Filter by rate value, e.g., 0.02 = show only if non-8h rate >= 0.02%
    show_top=SHOW_TOP,
    dark_mode=DARK_MODE
)

✅ UI Logic loaded.


⏰ 07:48:13
🏆 Top 3 Opps | Imbalance Rate ≥ 0.3%


Coin,Interval Imbalance,Exchanges,binance,bybit,kucoin
SIGN,binance:4h (-2.0000%) bybit:4h (-2.0000%),3,-2.0000%4h*,-2.0000%4h*,-2.0000%8h*
AXL,binance:4h (-1.1025%) bybit:4h (-1.2440%),3,-1.1025%4h*,-1.2440%4h,-1.8170%8h*
SIREN,binance:1h (-0.1255%) bybit:4h (-0.5546%),3,-0.1255%1h*,-0.5546%4h*,-0.3336%8h


In [3]:
# CELL 5: TIME TABLE
# ═══════════════════════════════════════════════════════════════════════════════

def build_funding_timeline(dark_mode=True, scale=0.7):
    """
    Creates a visual mini-map of funding rate settlement times with countdown (Sri Lanka Time)
    scale: 0.5 = 50% size, 0.7 = 70% size, 1.0 = 100% size (default)
    """
    from datetime import datetime, timezone, timedelta
    from IPython.display import display, HTML
    
    # Current UTC time
    now_utc = datetime.now(timezone.utc)
    
    # Convert to Sri Lanka Time (SLT = UTC+5:30)
    slt_offset = timedelta(hours=5, minutes=30)
    now_slt = now_utc + slt_offset
    
    current_hour = now_slt.hour
    current_minute = now_slt.minute
    current_second = now_slt.second
    
    def to_12h(hour, minute=0):
        period = "AM" if hour < 12 else "PM"
        display_hour = hour if hour <= 12 else hour - 12
        if display_hour == 0:
            display_hour = 12
        return f"{display_hour}:{minute:02d} {period}"
    
    def to_12h_short(hour):
        if hour == 0:
            return "12am"
        elif hour < 12:
            return f"{hour}am"
        elif hour == 12:
            return "12pm"
        else:
            return f"{hour - 12}pm"
    
    utc_schedules = {
        '1H': [h for h in range(24)],
        '4H': [0, 4, 8, 12, 16, 20],
        '8H': [0, 8, 16],
    }
    
    def utc_to_slt_hour(utc_hour):
        slt_hour = (utc_hour + 5) % 24
        return slt_hour
    
    schedules_slt = {
        '1H': [utc_to_slt_hour(h) for h in utc_schedules['1H']],
        '4H': sorted([utc_to_slt_hour(h) for h in utc_schedules['4H']]),
        '8H': sorted([utc_to_slt_hour(h) for h in utc_schedules['8H']]),
    }
    
    def get_countdown(utc_hours_list):
        for offset in range(0, 25):
            check_hour = (now_utc.hour + offset) % 24
            if check_hour in utc_hours_list:
                if offset == 0:
                    if now_utc.minute > 0 or now_utc.second > 0:
                        continue
                    else:
                        slt_hour = (check_hour + 5) % 24
                        return slt_hour, 30, "00:00:00"
                
                hours_left = offset - 1
                mins_left = 59 - now_utc.minute
                secs_left = 60 - now_utc.second
                if secs_left == 60:
                    secs_left = 0
                else:
                    if mins_left == 0 and secs_left > 0:
                        mins_left = 59
                        hours_left = max(0, hours_left - 1) if hours_left > 0 else 0
                    elif mins_left > 0:
                        mins_left -= 1
                
                slt_hour = (check_hour + 5) % 24
                return slt_hour, 30, f"{hours_left:02d}:{mins_left:02d}:{secs_left:02d}"
        return 5, 30, "00:00:00"
    
    next_1h_hour, next_1h_min, countdown_1h = get_countdown(utc_schedules['1H'])
    next_4h_hour, next_4h_min, countdown_4h = get_countdown(utc_schedules['4H'])
    next_8h_hour, next_8h_min, countdown_8h = get_countdown(utc_schedules['8H'])
    
    next_1h_str = to_12h(next_1h_hour, next_1h_min)
    next_4h_str = to_12h(next_4h_hour, next_4h_min)
    next_8h_str = to_12h(next_8h_hour, next_8h_min)
    
    current_time_str = to_12h(current_hour, current_minute).replace(
        f":{current_minute:02d}", 
        f":{current_minute:02d}:{current_second:02d}"
    )
    
    if dark_mode:
        bg_main, bg_card = '#1a1a1a', '#252525'
        border_color, text_main = '#30363d', '#e6e6e6'
        color_1h, color_4h, color_8h = '#4dabf7', '#51cf66', '#ffd43b'
        now_marker = '#ff6b6b'
    else:
        bg_main, bg_card = '#f8f9fa', '#ffffff'
        border_color, text_main = '#dee2e6', '#2c3e50'
        color_1h, color_4h, color_8h = '#1976d2', '#2e7d32', '#f57c00'
        now_marker = '#c62828'
    
    def build_timeline_row(label, slt_hours_list, color):
        dots_html = ""
        for h in range(24):
            if h == current_hour:
                if h in slt_hours_list:
                    dot = f'<div style="width:20px;height:20px;display:flex;align-items:center;justify-content:center;font-size:10px;color:{now_marker};font-weight:900;margin:2px;box-sizing:border-box;border:3px solid {color};border-radius:50%;background:{bg_card};">▼</div>'
                else:
                    dot = f'<div style="width:20px;height:20px;display:flex;align-items:center;justify-content:center;font-size:8px;color:{now_marker};font-weight:900;margin:2px;box-sizing:border-box;">▼</div>'
            elif h in slt_hours_list:
                dot = f'<div style="width:20px;height:20px;background:{color};border-radius:50%;margin:2px;box-sizing:border-box;opacity:0.9;"></div>'
            else:
                dot = f'<div style="width:20px;height:20px;background:{border_color};border-radius:50%;margin:2px;box-sizing:border-box;opacity:0.3;"></div>'
            dots_html += dot
        return dots_html
    
    hour_labels = "".join([
        f'<div style="width:20px;margin:2px;box-sizing:border-box;text-align:center;font-size:8px;color:{text_main};opacity:0.7;">{to_12h_short(h)}</div>' 
        for h in range(24)
    ])
    
    html = f"""
    <div style="display:inline-block;transform:scale({scale});transform-origin:top left;">
        <div style="background:{bg_main};border:2px solid {border_color};border-radius:10px;padding:15px;font-family:Consolas,monospace;">
            
            <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:15px;padding-bottom:10px;border-bottom:1px solid {border_color};">
                <span style="font-size:14px;font-weight:700;color:{text_main};">⏰ FUNDING SETTLEMENT TIMELINE</span>
                <span style="font-size:12px;color:{text_main};opacity:0.8;">{current_time_str} </span>
            </div>
            
            <div style="display:flex;gap:15px;margin-bottom:20px;">
                <div style="flex:1;background:{bg_card};border:1px solid {color_1h};border-radius:8px;padding:12px;text-align:center;">
                    <div style="font-size:11px;color:{color_1h};font-weight:600;">1H FUNDING</div>
                    <div style="font-size:20px;color:{text_main};font-weight:700;margin:5px 0;">{countdown_1h}</div>
                    <div style="font-size:10px;color:{text_main};opacity:0.6;">Next @ {next_1h_str}</div>
                </div>
                <div style="flex:1;background:{bg_card};border:1px solid {color_4h};border-radius:8px;padding:12px;text-align:center;">
                    <div style="font-size:11px;color:{color_4h};font-weight:600;">4H FUNDING</div>
                    <div style="font-size:20px;color:{text_main};font-weight:700;margin:5px 0;">{countdown_4h}</div>
                    <div style="font-size:10px;color:{text_main};opacity:0.6;">Next @ {next_4h_str}</div>
                </div>
                <div style="flex:1;background:{bg_card};border:1px solid {color_8h};border-radius:8px;padding:12px;text-align:center;">
                    <div style="font-size:11px;color:{color_8h};font-weight:600;">8H FUNDING</div>
                    <div style="font-size:20px;color:{text_main};font-weight:700;margin:5px 0;">{countdown_8h}</div>
                    <div style="font-size:10px;color:{text_main};opacity:0.6;">Next @ {next_8h_str}</div>
                </div>
            </div>
            
            <div style="background:{bg_card};border-radius:8px;padding:12px;border:1px solid {border_color};">
                <div style="font-size:11px;color:{text_main};margin-bottom:10px;font-weight:600;">24H TIMELINE (SLT - Settlements at 00:30)</div>
                
                <div style="display:flex;margin-bottom:5px;">
                    <span style="width:30px;margin-right:5px;"></span>
                    <div style="display:flex;">{hour_labels}</div>
                </div>
                
                <div style="display:flex;align-items:center;margin:3px 0;">
                    <span style="width:30px;margin-right:5px;font-size:10px;color:{color_1h};font-weight:600;">1H</span>
                    <div style="display:flex;">{build_timeline_row('1H', schedules_slt['1H'], color_1h)}</div>
                </div>
                
                <div style="display:flex;align-items:center;margin:3px 0;">
                    <span style="width:30px;margin-right:5px;font-size:10px;color:{color_4h};font-weight:600;">4H</span>
                    <div style="display:flex;">{build_timeline_row('4H', schedules_slt['4H'], color_4h)}</div>
                </div>
                
                <div style="display:flex;align-items:center;margin:3px 0;">
                    <span style="width:30px;margin-right:5px;font-size:10px;color:{color_8h};font-weight:600;">8H</span>
                    <div style="display:flex;">{build_timeline_row('8H', schedules_slt['8H'], color_8h)}</div>
                </div>
            </div>            
        </div>
    </div>
    """
    display(HTML(html))

build_funding_timeline(
     scale=1,
     dark_mode=DARK_MODE)

NameError: name 'DARK_MODE' is not defined

TEST3 Triangular arb (doesn't related to anything above)

In [25]:
import requests
import pandas as pd
from itertools import permutations

def get_gateio_tickers():
    url = "https://api.gateio.ws/api/v4/spot/tickers"
    r = requests.get(url, timeout=15)
    r.raise_for_status()
    return r.json()

def build_conversion_edges(tickers):
    """
    Build directed conversion rates.

    For pair BASE_QUOTE:
    - BASE -> QUOTE uses bid
    - QUOTE -> BASE uses 1 / ask
    """
    edges = {}

    for t in tickers:
        pair = t.get("currency_pair")
        if not pair or "_" not in pair:
            continue

        base, quote = pair.split("_")
        bid = float(t.get("highest_bid", 0) or 0)
        ask = float(t.get("lowest_ask", 0) or 0)

        if bid <= 0 or ask <= 0:
            continue

        # Sell base to get quote
        edges[(base, quote)] = {
            "rate": bid,
            "pair": pair,
            "side": f"sell {base} for {quote}"
        }

        # Use quote to buy base
        edges[(quote, base)] = {
            "rate": 1 / ask,
            "pair": pair,
            "side": f"buy {base} with {quote}"
        }

    return edges

def scan_triangular_arbitrage(start="USDT", coins=None, fee=0.001, start_amount=1000):
    """
    Scan simple triangular routes:
    start -> coin_a -> coin_b -> start
    """
    if coins is None:
        coins = ["BTC", "ETH", "SOL", "XRP", "DOGE", "ADA", "LTC", "TRX"]

    coins = [c for c in coins if c != start]

    tickers = get_gateio_tickers()
    edges = build_conversion_edges(tickers)

    rows = []

    for a, b in permutations(coins, 2):
        if (start, a) not in edges:
            continue
        if (a, b) not in edges:
            continue
        if (b, start) not in edges:
            continue

        e1 = edges[(start, a)]
        e2 = edges[(a, b)]
        e3 = edges[(b, start)]

        amt1 = start_amount * e1["rate"] * (1 - fee)
        amt2 = amt1 * e2["rate"] * (1 - fee)
        final_amt = amt2 * e3["rate"] * (1 - fee)

        profit = final_amt - start_amount
        profit_pct = (profit / start_amount) * 100

        rows.append({
            "route": f"{start} -> {a} -> {b} -> {start}",
            "final_amount": round(final_amt, 6),
            "profit": round(profit, 6),
            "profit_pct": round(profit_pct, 6),
            "step1_pair": e1["pair"],
            "step2_pair": e2["pair"],
            "step3_pair": e3["pair"],
        })

    df = pd.DataFrame(rows)

    if df.empty:
        return df

    return df.sort_values("profit_pct", ascending=False).reset_index(drop=True)

In [26]:
coins = ["BTC", "ETH", "SOL", "XRP", "DOGE", "ADA", "LTC", "TRX"]

df = scan_triangular_arbitrage(
    start="USDT",
    coins=coins,
    fee=0.001,        # 0.1% fee per trade
    start_amount=1000
)

df.head(20)

,route,final_amount,profit,profit_pct,step1_pair,step2_pair,step3_pair
0,USDT -> DOGE -> BTC -> USDT,997.989827,-2.010173,-0.201017,DOGE_USDT,DOGE_BTC,BTC_USDT
1,USDT -> ETH -> BTC -> USDT,996.813604,-3.186396,-0.318640,ETH_USDT,ETH_BTC,BTC_USDT
2,USDT -> ADA -> BTC -> USDT,996.780759,-3.219241,-0.321924,ADA_USDT,ADA_BTC,BTC_USDT
3,USDT -> BTC -> ETH -> USDT,996.709122,-3.290878,-0.329088,BTC_USDT,ETH_BTC,ETH_USDT
4,USDT -> XRP -> BTC -> USDT,996.666660,-3.333340,-0.333334,XRP_USDT,XRP_BTC,BTC_USDT
5,USDT -> LTC -> BTC -> USDT,996.403075,-3.596925,-0.359693,LTC_USDT,LTC_BTC,BTC_USDT
6,USDT -> BTC -> XRP -> USDT,996.089686,-3.910314,-0.391031,BTC_USDT,XRP_BTC,XRP_USDT
7,USDT -> BTC -> LTC -> USDT,995.241513,-4.758487,-0.475849,BTC_USDT,LTC_BTC,LTC_USDT
8,USDT -> BTC -> ADA -> USDT,995.104607,-4.895393,-0.489539,BTC_USDT,ADA_BTC,ADA_USDT
9,USDT -> BTC -> DOGE -> USDT,995.020773,-4.979227,-0.497923,BTC_USDT,DOGE_BTC,DOGE_USDT
